# Homework: Inventory Allocation 

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2023, University of Chicago 

A retailer allocates units of a product at the start of each period to 6 retail stores.  Based on historical data, we have gathered 10 independent demand figures (scenarios) for a single period for each store.  These scenarios for each store are considered independent of each other and independent between one store and another.  Furthermore, each scenario is equally likely to occur.  

Each store also has its own per-unit penalties for stocking more units than demanded (Excess Penalty) or fewer units than demanded (Shortage Penalty).  Units left over one period are discarded (part of the Excess Penalty), and demand that exceeds supply is not back-ordered, but is lost (part of the Shortage Penalty).

The demand scenarios and excess and shortage penalties are given in the table below:

| Store | Excess Penalty per unit   | Shortage Penalty per unit | demand scenarios | 
| :--------: | ----------: | ------: | ----: | 
| A  | 10 | 10    | [100, 120, 100, 90, 80, 60, 70, 130, 110, 105] |
| B  | 20 | 15    | [80, 70, 50, 40, 50, 60, 90, 120, 85, 45] | 
| C  | 10 | 5 | [130, 150, 120, 160, 80, 170, 90, 85, 75, 105] | 
| D  | 5  | 5    | [25, 35, 50, 65, 45, 35, 70, 50, 40, 80] | 
| E  | 15  | 15    | [105, 95, 100, 90, 85, 110, 95, 100, 80, 100] | 
| F  | 15 | 10    | [100, 120, 100, 90, 80, 60, 70, 130, 110, 105] | 

The retailer seeks a policy to allocate exactly 500 units between the stores at the start of each period.  The policy will send the same store the same quantity every period.

Formulate and solve a 2-stage stochastic optimization model to determine how many integer units the retailer should send to each store at the start of each period to minimize the expected excess and shortage penalty costs.   

In [15]:
# === SETUP ===
from pulp import *
import os
from pprint import pprint

# Portable solver setup, to allow work locally (ARM64 architecture) as well as in JupyterHub
from pulp import COIN_CMD
if os.path.exists('/opt/homebrew/opt/cbc/bin/cbc'):
    solver = COIN_CMD(path='/opt/homebrew/opt/cbc/bin/cbc', msg=0)
else:
    solver = pulp.PULP_CBC_CMD(msg=0)

**Here is a var to store `total_units` and a list you can use called `stores`.** 

In [4]:
total_units = 500

stores = [
    {'name': 'A', 
     'excess_penalty': 10,
     'shortage_penalty': 10,
     'scenario_demands': [100, 120, 100, 90, 80, 60, 70, 130, 110, 105]
    }, 
    {'name': 'B', 
     'excess_penalty': 20,
     'shortage_penalty': 15,
     'scenario_demands': [80, 70, 50, 40, 50, 60, 90, 120, 85, 45]
    }, 
    {'name': 'C', 
     'excess_penalty': 10,
     'shortage_penalty': 5,
     'scenario_demands': [130, 150, 120, 160, 80, 170, 90, 85, 75, 105]
    }, 
    {'name': 'D', 
     'excess_penalty': 5,
     'shortage_penalty': 5,
     'scenario_demands': [25, 35, 50, 65, 45, 35, 70, 50, 40, 80]
    }, 
    {'name': 'E', 
     'excess_penalty': 15,
     'shortage_penalty': 15,
     'scenario_demands': [105, 95, 100, 90, 85, 110, 95, 100, 80, 100]
    }, 
    {'name': 'F', 
     'excess_penalty': 15,
     'shortage_penalty': 10,
     'scenario_demands': [100, 120, 100, 90, 80, 60, 70, 130, 110, 105]
    }
]

## Your Solution


**1. In broad terms, what are the variables, objective and constraints of this problem? You don't need to list the entire formulation. Just describe the structure/characteristics of your model.**

markdown## 2-Stage Stochastic Programming Formulation

**Sets:**
- Stores ($s = 1, 2, 3, 4, 5, 6$ corresponding to A, B, C, D, E, F)
- Scenarios ($\omega = 1, 2, 3, \dots, 10$)

**Parameters:**
- $D_{s,\omega}$: demand at store $s$ in scenario $\omega$
- $e_s$: excess penalty per unit at store $s$
- $h_s$: shortage penalty per unit at store $s$
- $Q = 500$: total units available for allocation
- $p_\omega = 0.1$: probability of scenario $\omega$ (equally likely)

**Decision Variables:**

*First-stage:*
- $x_s \in \mathbb{Z} \quad \geq 0$: integer units allocated to store $s$

*Second-stage:*
- $y^+_{s,\omega} \geq 0$: excess units at store $s$ in scenario $\omega$
- $y^-_{s,\omega} \geq 0$: shortage units at store $s$ in scenario $\omega$

**Objective:**

Minimize expected total penalty cost:

$$
\min \sum_{\omega=1}^{10} p_\omega \sum_{s=1}^{6} \left( e_s \cdot y^+_{s,\omega} + h_s \cdot y^-_{s,\omega} \right)
$$

Since $p_\omega = 0.1$ for all scenarios:

$$
\min \frac{1}{10} \sum_{\omega=1}^{10} \sum_{s=1}^{6} \left( e_s \cdot y^+_{s,\omega} + h_s \cdot y^-_{s,\omega} \right)
$$

Constraints:

**1. Total allocation constraint:**

$$
\sum_{s=1}^{6} x_s = Q
$$

**2. For each store $s$ and scenario $\omega$, balance excess and shortage with allocation-demand difference:**
$$
y^+_{s,\omega} - y^-_{s,\omega} = x_s - D_{s,\omega} \quad \forall\ s = 1, \dots, 6, \quad \forall\ \omega = 1, \dots, 10
$$

**3. Non-negativity:**
$$
x_s \geq 0, \quad y^+_{s,\omega} \geq 0, \quad y^-_{s,\omega} \geq 0
$$

**2. Create a PuLP LpProblem object and store it in the variable `model`.** 

In [7]:
model = LpProblem("Inventory_Allocation",LpMinimize)

**3. Create nicely named dictionaries to store your Stage 1 and Stage 2 variables.** 

In [8]:
# First-stage variables: allocation to each store
x = {}
for s in range(len(stores)):
    store_name = stores[s]['name']
    x[s] = LpVariable(f"x_{store_name}", lowBound=0, cat='Integer')

# Second-stage variables: excess and shortage for each store-scenario combination
y_excess = {}
y_shortage = {}
num_scenarios = len(stores[0]['scenario_demands'])

for s in range(len(stores)):
    store_name = stores[s]['name']
    for omega in range(num_scenarios):
        y_excess[(s, omega)] = LpVariable(f"y_excess_{store_name}_{omega}", lowBound=0, cat='Continuous')
        y_shortage[(s, omega)] = LpVariable(f"y_shortage_{store_name}_{omega}", lowBound=0, cat='Continuous')

print("Allocation variables:", x)
print("Excess variables:", y_excess)
print("Shortage variables:", y_shortage)

Allocation variables: {0: x_A, 1: x_B, 2: x_C, 3: x_D, 4: x_E, 5: x_F}
Excess variables: {(0, 0): y_excess_A_0, (0, 1): y_excess_A_1, (0, 2): y_excess_A_2, (0, 3): y_excess_A_3, (0, 4): y_excess_A_4, (0, 5): y_excess_A_5, (0, 6): y_excess_A_6, (0, 7): y_excess_A_7, (0, 8): y_excess_A_8, (0, 9): y_excess_A_9, (1, 0): y_excess_B_0, (1, 1): y_excess_B_1, (1, 2): y_excess_B_2, (1, 3): y_excess_B_3, (1, 4): y_excess_B_4, (1, 5): y_excess_B_5, (1, 6): y_excess_B_6, (1, 7): y_excess_B_7, (1, 8): y_excess_B_8, (1, 9): y_excess_B_9, (2, 0): y_excess_C_0, (2, 1): y_excess_C_1, (2, 2): y_excess_C_2, (2, 3): y_excess_C_3, (2, 4): y_excess_C_4, (2, 5): y_excess_C_5, (2, 6): y_excess_C_6, (2, 7): y_excess_C_7, (2, 8): y_excess_C_8, (2, 9): y_excess_C_9, (3, 0): y_excess_D_0, (3, 1): y_excess_D_1, (3, 2): y_excess_D_2, (3, 3): y_excess_D_3, (3, 4): y_excess_D_4, (3, 5): y_excess_D_5, (3, 6): y_excess_D_6, (3, 7): y_excess_D_7, (3, 8): y_excess_D_8, (3, 9): y_excess_D_9, (4, 0): y_excess_E_0, (4, 1): 

**4. Add your objective function to your `model`.**

In [9]:
# Objective: Minimize expected total penalty cost
model += (1/num_scenarios) * lpSum([
    stores[s]['excess_penalty'] * y_excess[(s, omega)] + 
    stores[s]['shortage_penalty'] * y_shortage[(s, omega)]
    for s in range(len(stores))
    for omega in range(num_scenarios)
]), "Expected_Total_Penalty_Cost"

**5. Add the constraints to your `model`.**

In [10]:
# Constraint 1: Total allocation must equal available units
model += lpSum([x[s] for s in range(len(stores))]) == total_units, "Total_Allocation"

# Constraint 2: Balance excess and shortage with allocation-demand difference for each store and scenario
for s in range(len(stores)):
    for omega in range(num_scenarios):
        demand = stores[s]['scenario_demands'][omega]
        model += (y_excess[(s, omega)] - y_shortage[(s, omega)] == x[s] - demand), \
                 f"Balance_Store_{stores[s]['name']}_Scenario_{omega}"

**6. Print your `model` and verify that the variables, objective and constraints all look correct.**

In [11]:
print(model)

Inventory_Allocation:
MINIMIZE
1.0*y_excess_A_0 + 1.0*y_excess_A_1 + 1.0*y_excess_A_2 + 1.0*y_excess_A_3 + 1.0*y_excess_A_4 + 1.0*y_excess_A_5 + 1.0*y_excess_A_6 + 1.0*y_excess_A_7 + 1.0*y_excess_A_8 + 1.0*y_excess_A_9 + 2.0*y_excess_B_0 + 2.0*y_excess_B_1 + 2.0*y_excess_B_2 + 2.0*y_excess_B_3 + 2.0*y_excess_B_4 + 2.0*y_excess_B_5 + 2.0*y_excess_B_6 + 2.0*y_excess_B_7 + 2.0*y_excess_B_8 + 2.0*y_excess_B_9 + 1.0*y_excess_C_0 + 1.0*y_excess_C_1 + 1.0*y_excess_C_2 + 1.0*y_excess_C_3 + 1.0*y_excess_C_4 + 1.0*y_excess_C_5 + 1.0*y_excess_C_6 + 1.0*y_excess_C_7 + 1.0*y_excess_C_8 + 1.0*y_excess_C_9 + 0.5*y_excess_D_0 + 0.5*y_excess_D_1 + 0.5*y_excess_D_2 + 0.5*y_excess_D_3 + 0.5*y_excess_D_4 + 0.5*y_excess_D_5 + 0.5*y_excess_D_6 + 0.5*y_excess_D_7 + 0.5*y_excess_D_8 + 0.5*y_excess_D_9 + 1.5*y_excess_E_0 + 1.5*y_excess_E_1 + 1.5*y_excess_E_2 + 1.5*y_excess_E_3 + 1.5*y_excess_E_4 + 1.5*y_excess_E_5 + 1.5*y_excess_E_6 + 1.5*y_excess_E_7 + 1.5*y_excess_E_8 + 1.5*y_excess_E_9 + 1.5*y_excess_F_0 + 

**7. Solve your `model` and print the optimal objective value.**

In [16]:
model.solve(solver)
print(f"Status: {LpStatus[model.status]}")

Status: Optimal


**8. Print out the optimal number of units allocated to each store.**

In [19]:
for s, var in x.items():
    print(f"{var.name} = {var.varValue}")

x_A = 100.0
x_B = 60.0
x_C = 90.0
x_D = 50.0
x_E = 100.0
x_F = 100.0
